<a href="https://colab.research.google.com/github/CHENXY2026/Diabetes-Drug-Recommendation-Based-on-Neural-Collaborative-Filtering/blob/main/%E5%9F%BA%E4%BA%8E%E7%A5%9E%E7%BB%8F%E5%8D%8F%E5%90%8C%E8%BF%87%E6%BB%A4%E7%9A%84%E7%B3%96%E5%B0%BF%E7%97%85%E8%8D%AF%E7%89%A9%E6%8E%A8%E8%8D%90.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve
import tensorflow as tf
from tensorflow.keras.models import Model, load_model, save_model
from tensorflow.keras.layers import Input, Embedding, Flatten, Dense, Concatenate, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import logging
import os
import warnings
warnings.filterwarnings('ignore')

# 设置日志
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# 设置随机种子，确保结果可复现
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# 全局变量 - 诊断类别映射
DIAGNOSIS_CATEGORY_MAP = {
    'missing': 0,
    'diabetes': 1,
    'cardiovascular': 2,
    'respiratory': 3,
    'digestive': 4,
    'genitourinary': 5,
    'musculoskeletal': 6,
    'injury': 7,
    'anemia': 8,
    'mental': 9,
    'other': 10
}

# 药物特征列表
DRUG_FEATURES = [
    'metformin', 'repaglinide', 'nateglinide', 'glimepiride', 'glipizide',
    'glyburide', 'pioglitazone', 'rosiglitazone', 'insulin',
    'glyburide-metformin', 'acarbose'
]

# 选定的患者特征
SELECTED_FEATURES = [
    'patient_nbr', 'A1Cresult', 'max_glu_serum', 'diag_1', 'diag_2', 'diag_3',
    'number_diagnoses', 'age', 'change', 'diabetesMed', 'num_medications',
    'time_in_hospital'
]

# 用于模型的临床特征
CLINICAL_FEATURES = [
    'A1Cresult', 'max_glu_serum',
    'diag_1_category', 'diag_2_category', 'diag_3_category',
    'number_diagnoses', 'age_value', 'change', 'diabetesMed',
    'num_medications', 'time_in_hospital'
]

# 创建输出目录
OUTPUT_DIR = 'diabetes_recommendation_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/plots', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/models', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/data', exist_ok=True)

def get_icd9_category(code):
    """将ICD-9代码转换为疾病类别，使用标准医学分类"""
    if pd.isna(code) or code == '':
        return 'missing'

    try:
        # 转换为数值
        code_num = float(str(code).split('.')[0])  # 去掉小数点后的部分

        # 使用标准医学分类
        if 250 <= code_num < 251:
            return 'diabetes'
        elif 390 <= code_num < 460:
            return 'cardiovascular'
        elif 460 <= code_num < 520:
            return 'respiratory'
        elif 520 <= code_num < 580:
            return 'digestive'
        elif 580 <= code_num < 630:
            return 'genitourinary'
        elif 710 <= code_num < 740:
            return 'musculoskeletal'
        elif 800 <= code_num < 1000:
            return 'injury'
        elif 280 <= code_num < 290:
            return 'anemia'
        elif 290 <= code_num < 320:
            return 'mental'
        else:
            return 'other'
    except (ValueError, TypeError):
        return 'missing'

def process_diagnosis_codes(df):
    """医学诊断代码处理方法 - 将ICD-9代码转换为有意义的医学类别"""
    # 为三个诊断字段创建类别
    for col in ['diag_1', 'diag_2', 'diag_3']:
        df[f'{col}_category'] = df[col].apply(get_icd9_category)
        # 将类别名称映射为数值
        df[f'{col}_category'] = df[f'{col}_category'].map(DIAGNOSIS_CATEGORY_MAP)

    return df

def safe_convert_to_numeric(value, default=0):
    """安全转换为数值类型，处理异常情况"""
    try:
        if pd.isna(value) or value == '':
            return default
        return float(value)
    except (ValueError, TypeError):
        return default

def download_and_load_data():
    """下载并加载数据集"""
    logger.info("下载并加载数据集...")
    try:
        # 读取数据
        df = pd.read_csv('/content/drive/MyDrive/diabetic data.csv')
        logger.info(f"数据集加载成功，形状: {df.shape}")
        return df
    except Exception as e:
        logger.error(f"数据集加载失败: {e}")
        raise

def preprocess_data(df, selected_features=SELECTED_FEATURES, drug_features=DRUG_FEATURES):
    """数据预处理主函数"""
    logger.info("开始数据预处理...")

    # 选择需要的列
    all_features = selected_features + drug_features
    df_selected = df[all_features].copy()

    # 处理 A1Cresult 和 max_glu_serum
    a1c_map = {'None': 0, 'Norm': 1, '>7': 2, '>8': 3}
    df_selected['A1Cresult'] = df_selected['A1Cresult'].map(a1c_map).fillna(0)

    glu_map = {'None': 0, 'Norm': 1, '>200': 2, '>300': 3}
    df_selected['max_glu_serum'] = df_selected['max_glu_serum'].map(glu_map).fillna(0)

    # 处理诊断代码 - 确保为数值型
    for col in ['diag_1', 'diag_2', 'diag_3']:
        # 删除非数字字符
        df_selected[col] = df_selected[col].astype(str).str.replace('[^0-9.]', '', regex=True)
        # 将空字符串转换为NaN
        df_selected[col] = df_selected[col].replace('', np.nan)
        # 转换为浮点数并填充缺失值
        df_selected[col] = pd.to_numeric(df_selected[col], errors='coerce').fillna(0)

    # 应用诊断代码处理函数
    df_selected = process_diagnosis_codes(df_selected)

    # 处理年龄分组
    age_map = {'[0-10)': 5, '[10-20)': 15, '[20-30)': 25, '[30-40)': 35,
               '[40-50)': 45, '[50-60)': 55, '[60-70)': 65, '[70-80)': 75,
               '[80-90)': 85, '[90-100)': 95}
    df_selected['age_value'] = df_selected['age'].map(age_map)

    # 处理 change 和 diabetesMed
    df_selected['change'] = df_selected['change'].map({'No': 0, 'Ch': 1}).fillna(0)
    df_selected['diabetesMed'] = df_selected['diabetesMed'].map({'No': 0, 'Yes': 1}).fillna(0)

    # 处理药物特征 - 统一编码为使用状态
    for drug in drug_features:
        drug_map = {'No': 0, 'Steady': 1, 'Up': 1, 'Down': 1}
        df_selected[drug] = df_selected[drug].map(drug_map).fillna(0)

    # 确定每位患者使用的药物数量
    df_selected['drugs_used'] = df_selected[drug_features].sum(axis=1)

    # 只保留至少使用过一种药物的患者
    df_filtered = df_selected[df_selected['drugs_used'] > 0].copy()

    # 删除患者重复记录，保留第一条记录
    df_filtered = df_filtered.drop_duplicates(subset=['patient_nbr'], keep='first')

    logger.info(f"预处理完成，筛选后数据集大小: {df_filtered.shape}")
    return df_filtered

def create_interaction_matrix(df, drug_features=DRUG_FEATURES):
    """创建用户-药物交互矩阵"""
    logger.info("创建用户-药物交互矩阵...")

    interactions = []

    for _, row in df.iterrows():
        patient_id = row['patient_nbr']

        for drug in drug_features:
            # 如果患者使用了该药物
            if row[drug] > 0:
                interactions.append({
                    'patient_nbr': patient_id,
                    'drug': drug,
                    'interaction': 1
                })
            else:
                interactions.append({
                    'patient_nbr': patient_id,
                    'drug': drug,
                    'interaction': 0
                })

    interactions_df = pd.DataFrame(interactions)
    logger.info(f"交互矩阵创建完成，形状: {interactions_df.shape}")
    return interactions_df

def encode_features(df, interactions_df, clinical_features=CLINICAL_FEATURES):
    """编码特征并准备模型输入"""
    logger.info("编码特征并准备模型输入...")

    # 编码患者ID和药物名称
    patient_encoder = LabelEncoder()
    drug_encoder = LabelEncoder()

    interactions_df['patient_id'] = patient_encoder.fit_transform(interactions_df['patient_nbr'])
    interactions_df['drug_id'] = drug_encoder.fit_transform(interactions_df['drug'])

    # 获取患者特征
    patient_features = df.copy()

    # 添加患者ID编码
    patient_features['patient_id'] = patient_encoder.transform(patient_features['patient_nbr'])

    # 标准化数值特征
    numerical_features = ['number_diagnoses', 'num_medications', 'time_in_hospital']
    scaler = StandardScaler()
    patient_features[numerical_features] = scaler.fit_transform(patient_features[numerical_features])

    # 选择最终用于模型的特征
    final_features = ['patient_id'] + clinical_features
    patient_features_encoded = patient_features[final_features].copy()

    # 保存编码器和标准化器，以便后续使用
    encoders = {
        'patient_encoder': patient_encoder,
        'drug_encoder': drug_encoder,
        'scaler': scaler
    }

    # 返回编码后的数据和编码器
    return interactions_df, patient_features_encoded, encoders

def prepare_model_inputs(interactions_df, patient_features_encoded):
    """准备模型输入数据"""
    logger.info("准备模型输入数据...")

    # 分割训练集和测试集
    train_interactions, test_interactions = train_test_split(
        interactions_df, test_size=0.2, random_state=RANDOM_SEED, stratify=interactions_df['interaction']
    )

    # 进一步分割出验证集
    train_interactions, val_interactions = train_test_split(
        train_interactions, test_size=0.15, random_state=RANDOM_SEED, stratify=train_interactions['interaction']
    )

    # 准备训练数据
    X_train_patient = train_interactions['patient_id'].values
    X_train_drug = train_interactions['drug_id'].values
    y_train = train_interactions['interaction'].values

    # 准备验证数据
    X_val_patient = val_interactions['patient_id'].values
    X_val_drug = val_interactions['drug_id'].values
    y_val = val_interactions['interaction'].values

    # 准备测试数据
    X_test_patient = test_interactions['patient_id'].values
    X_test_drug = test_interactions['drug_id'].values
    y_test = test_interactions['interaction'].values

    # 准备患者特征
    def get_patient_features(patient_ids, features_df):
        patient_features = []
        for patient_id in patient_ids:
            features = features_df[features_df['patient_id'] == patient_id].iloc[0, 1:].values
            patient_features.append(features)
        return np.array(patient_features)

    X_train_patient_features = get_patient_features(X_train_patient, patient_features_encoded)
    X_val_patient_features = get_patient_features(X_val_patient, patient_features_encoded)
    X_test_patient_features = get_patient_features(X_test_patient, patient_features_encoded)

    train_data = {
        'patient_ids': X_train_patient,
        'drug_ids': X_train_drug,
        'patient_features': X_train_patient_features,
        'labels': y_train
    }

    val_data = {
        'patient_ids': X_val_patient,
        'drug_ids': X_val_drug,
        'patient_features': X_val_patient_features,
        'labels': y_val
    }

    test_data = {
        'patient_ids': X_test_patient,
        'drug_ids': X_test_drug,
        'patient_features': X_test_patient_features,
        'labels': y_test
    }

    return train_data, val_data, test_data

def analyze_data(df, interactions_df):
    """分析数据并生成描述性统计"""
    logger.info("生成数据分析报告...")

    # 1. 基本统计信息
    print("数据集基本信息:")
    print(f"总患者数: {df['patient_nbr'].nunique()}")
    print(f"总记录数: {len(df)}")

    # 2. 药物使用情况
    drug_usage = {}
    for drug in DRUG_FEATURES:
        drug_usage[drug] = df[drug].sum()

    drug_usage_df = pd.DataFrame({
        'drug': list(drug_usage.keys()),
        'usage_count': list(drug_usage.values())
    }).sort_values('usage_count', ascending=False)

    print("\n药物使用情况:")
    print(drug_usage_df)

    # 3. 诊断类别分布
    diag_categories = {v: k for k, v in DIAGNOSIS_CATEGORY_MAP.items()}

    for col in ['diag_1_category', 'diag_2_category', 'diag_3_category']:
        print(f"\n{col} 分布:")
        category_counts = df[col].value_counts().sort_index()
        for category_id, count in category_counts.items():
            category_name = diag_categories.get(category_id, 'Unknown')
            print(f"{category_name}: {count} ({count/len(df):.2%})")

    # 4. 药物交互情况
    interaction_stats = interactions_df['interaction'].value_counts()
    print("\n药物交互情况:")
    print(f"有交互: {interaction_stats.get(1, 0)}")
    print(f"无交互: {interaction_stats.get(0, 0)}")
    print(f"交互率: {interaction_stats.get(1, 0) / len(interactions_df):.2%}")

    # 5. 可视化药物使用分布
    plt.figure(figsize=(12, 6))
    sns.barplot(x='drug', y='usage_count', data=drug_usage_df)
    plt.title('药物使用频率分布')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/plots/drug_usage_distribution.png')
    plt.close()

    # 6. 可视化年龄分布
    plt.figure(figsize=(10, 6))
    sns.countplot(x='age', data=df)
    plt.title('患者年龄分布')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/plots/age_distribution.png')
    plt.close()

    # 7. 可视化诊断类别与药物使用关系
    plt.figure(figsize=(14, 10))

    # 创建一个新的DataFrame，将诊断类别ID转换为名称
    df_plot = df.copy()
    df_plot['diag_1_name'] = df_plot['diag_1_category'].map(diag_categories)

    # 计算每个诊断类别的药物使用率
    diagnosis_drug_matrix = df_plot.groupby('diag_1_name')[DRUG_FEATURES].mean()

    # 绘制热图
    sns.heatmap(diagnosis_drug_matrix, annot=True, cmap='YlGnBu', fmt='.2f')
    plt.title('主要诊断类别与药物使用关系热图')
    plt.xlabel('药物')
    plt.ylabel('诊断类别')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/plots/diagnosis_drug_heatmap.png')
    plt.close()

    # 8. 诊断类别分布饼图
    plt.figure(figsize=(10, 10))
    df_plot['diag_1_name'].value_counts().plot(
        kind='pie', autopct='%1.1f%%', startangle=90)
    plt.title('主要诊断类别分布')
    plt.ylabel('')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/plots/diagnosis_distribution_pie.png')
    plt.close()

    # 9. 相关性分析
    plt.figure(figsize=(12, 10))
    correlation_features = CLINICAL_FEATURES + DRUG_FEATURES
    correlation_matrix = df[correlation_features].corr()
    mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
    sns.heatmap(correlation_matrix, mask=mask, cmap='coolwarm', annot=False,
                center=0, square=True, linewidths=.5)
    plt.title('特征相关性热图')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/plots/feature_correlation_heatmap.png')
    plt.close()

    logger.info("数据分析报告生成完成")

def build_ncf_model(n_patients, n_drugs, n_factors, n_patient_features):
    """构建NCF模型"""
    logger.info("构建NCF模型...")

    # 输入层
    patient_input = Input(shape=(1,), name='patient_input')
    drug_input = Input(shape=(1,), name='drug_input')
    patient_features_input = Input(shape=(n_patient_features,), name='patient_features_input')

    # 嵌入层
    patient_embedding = Embedding(n_patients, n_factors, name='patient_embedding')(patient_input)
    drug_embedding = Embedding(n_drugs, n_factors, name='drug_embedding')(drug_input)

    # 展平嵌入
    patient_vec = Flatten()(patient_embedding)
    drug_vec = Flatten()(drug_embedding)

    # 连接嵌入和患者特征
    concat = Concatenate()([patient_vec, drug_vec, patient_features_input])

    # 全连接层
    fc1 = Dense(128, activation='relu')(concat)
    fc1 = Dropout(0.3)(fc1)
    fc2 = Dense(64, activation='relu')(fc1)
    fc2 = Dropout(0.3)(fc2)
    fc3 = Dense(32, activation='relu')(fc2)
    fc3 = Dropout(0.3)(fc3)

    # 输出层
    output = Dense(1, activation='sigmoid')(fc3)

    # 编译模型
    model = Model([patient_input, drug_input, patient_features_input], output)
    model.compile(optimizer=Adam(learning_rate=0.001),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

    return model

def train_model(model, train_data, val_data, epochs=30, batch_size=128):
    """训练NCF模型"""
    logger.info("开始训练模型...")

    # 设置回调函数
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=0.00001),
        ModelCheckpoint(filepath=f'{OUTPUT_DIR}/models/best_model.h5',
                        monitor='val_loss', save_best_only=True)
    ]

    # 训练模型
    history = model.fit(
        [train_data['patient_ids'], train_data['drug_ids'], train_data['patient_features']],
        train_data['labels'],
        batch_size=batch_size,
        epochs=epochs,
        validation_data=(
            [val_data['patient_ids'], val_data['drug_ids'], val_data['patient_features']],
            val_data['labels']
        ),
        callbacks=callbacks,
        verbose=1
    )

    return model, history

def evaluate_model(model, test_data):
    """评估模型性能"""
    logger.info("评估模型性能...")

    # 评估模型
    test_loss, test_accuracy = model.evaluate(
        [test_data['patient_ids'], test_data['drug_ids'], test_data['patient_features']],
        test_data['labels'],
        verbose=1
    )

    # 预测测试集
    y_pred_prob = model.predict([test_data['patient_ids'], test_data['drug_ids'], test_data['patient_features']])
    y_pred = (y_pred_prob > 0.5).astype(int).flatten()

    # 计算评估指标
    cm = confusion_matrix(test_data['labels'], y_pred)
    print("混淆矩阵:")
    print(cm)

    print("\n分类报告:")
    print(classification_report(test_data['labels'], y_pred))

    # ROC-AUC
    auc = roc_auc_score(test_data['labels'], y_pred_prob)
    print(f"ROC-AUC: {auc:.4f}")

    # 绘制ROC曲线
    from sklearn.metrics import roc_curve

    fpr, tpr, _ = roc_curve(test_data['labels'], y_pred_prob)
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, label=f'AUC = {auc:.4f}')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('假阳性率')
    plt.ylabel('真阳性率')
    plt.title('ROC曲线')
    plt.legend()
    plt.savefig(f'{OUTPUT_DIR}/plots/roc_curve.png')
    plt.close()

    # 绘制精确率-召回率曲线
    precision, recall, _ = precision_recall_curve(test_data['labels'], y_pred_prob)
    plt.figure(figsize=(8, 6))
    plt.plot(recall, precision)
    plt.xlabel('召回率')
    plt.ylabel('精确率')
    plt.title('精确率-召回率曲线')
    plt.savefig(f'{OUTPUT_DIR}/plots/precision_recall_curve.png')
    plt.close()

    # 绘制混淆矩阵
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('混淆矩阵')
    plt.xlabel('预测标签')
    plt.ylabel('真实标签')
    plt.savefig(f'{OUTPUT_DIR}/plots/confusion_matrix.png')
    plt.close()

    # 返回评估结果
    evaluation_results = {
        'test_loss': test_loss,
        'test_accuracy': test_accuracy,
        'auc': auc,
        'confusion_matrix': cm,
        'y_pred': y_pred,
        'y_pred_prob': y_pred_prob
    }

    return evaluation_results

def visualize_training_history(history):
    """可视化训练历史"""
    logger.info("可视化训练历史...")

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'], label='训练损失')
    plt.plot(history.history['val_loss'], label='验证损失')
    plt.title('模型损失')
    plt.xlabel('Epoch')
    plt.ylabel('损失')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.history['accuracy'], label='训练准确率')
    plt.plot(history.history['val_accuracy'], label='验证准确率')
    plt.title('模型准确率')
    plt.xlabel('Epoch')
    plt.ylabel('准确率')
    plt.legend()

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/plots/training_history.png')
    plt.close()

def recommend_drugs_for_patient(patient_id, model, encoders, patient_features_df, top_n=5):
    """为特定患者生成药物推荐"""
    # 获取患者特征
    patient_idx = encoders['patient_encoder'].transform([patient_id])[0]
    patient_features = patient_features_df[patient_features_df['patient_id'] == patient_idx].iloc[0, 1:].values

    # 为每种药物预测交互概率
    drug_ids = np.arange(len(encoders['drug_encoder'].classes_))
    patient_ids = np.full_like(drug_ids, patient_idx)
    patient_features_array = np.tile(patient_features, (len(drug_ids), 1))

    # 预测
    predictions = model.predict([patient_ids, drug_ids, patient_features_array])

    # 获取药物名称
    drug_names = encoders['drug_encoder'].inverse_transform(drug_ids)

    # 创建推荐数据框
    recommendations = pd.DataFrame({
        'drug': drug_names,
        'probability': predictions.flatten()
    })

    # 排序并返回前N个推荐
    return recommendations.sort_values('probability', ascending=False).head(top_n)

def visualize_recommendations(model, encoders, patient_features_df, df, n_patients=5):
    """为多个患者可视化药物推荐"""
    logger.info("可视化药物推荐...")

    # 随机选择几个患者
    sample_patients = np.random.choice(df['patient_nbr'].unique(), n_patients, replace=False)

    plt.figure(figsize=(15, 4*n_patients))
    for i, patient_id in enumerate(sample_patients):
        recommendations = recommend_drugs_for_patient(
            patient_id, model, encoders, patient_features_df
        )

        plt.subplot(n_patients, 1, i+1)
        sns.barplot(x='probability', y='drug', data=recommendations)
        plt.title(f'患者 {patient_id} 的药物推荐')
        plt.xlabel('推荐概率')
        plt.ylabel('药物')
        plt.xlim(0, 1)

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/plots/patient_recommendations.png')
    plt.close()

def generate_recommendations_for_new_patient(patient_features_dict, model, encoders, clinical_features=CLINICAL_FEATURES, top_n=5):
    """为新患者生成药物推荐"""
    # 创建特征向量
    features = []
    for feature in clinical_features:
        if feature in patient_features_dict:
            features.append(patient_features_dict[feature])
        else:
            # 如果缺少特征，使用0
            features.append(0)

    features = np.array(features).reshape(1, -1)

    # 为每种药物预测交互概率
    drug_ids = np.arange(len(encoders['drug_encoder'].classes_))
    patient_ids = np.zeros_like(drug_ids)  # 新患者使用ID 0
    patient_features_array = np.tile(features, (len(drug_ids), 1))

    # 预测
    predictions = model.predict([patient_ids, drug_ids, patient_features_array])

    # 获取药物名称
    drug_names = encoders['drug_encoder'].inverse_transform(drug_ids)

    # 创建推荐数据框
    recommendations = pd.DataFrame({
        'drug': drug_names,
        'probability': predictions.flatten()
    })

    # 排序并返回前N个推荐
    return recommendations.sort_values('probability', ascending=False).head(top_n)

def save_model_and_artifacts(model, encoders, clinical_features=CLINICAL_FEATURES):
    """保存模型和相关工件"""
    logger.info("保存模型和相关工件...")

    # 保存模型
    model.save(f'{OUTPUT_DIR}/models/diabetes_recommendation_model.h5')

    # 保存编码器
    import pickle
    with open(f'{OUTPUT_DIR}/models/encoders.pkl', 'wb') as f:
        pickle.dump(encoders, f)

    # 保存特征列表
    with open(f'{OUTPUT_DIR}/models/clinical_features.txt', 'w') as f:
        for feature in clinical_features:
            f.write(f"{feature}\n")

    logger.info("模型和工件保存完成")

def load_model_and_artifacts():
    """加载模型和相关工件"""
    logger.info("加载模型和相关工件...")

    # 加载模型
    model = load_model(f'{OUTPUT_DIR}/models/diabetes_recommendation_model.h5')

    # 加载编码器
    import pickle
    with open(f'{OUTPUT_DIR}/models/encoders.pkl', 'rb') as f:
        encoders = pickle.load(f)

    # 加载特征列表
    clinical_features = []
    with open(f'{OUTPUT_DIR}/models/clinical_features.txt', 'r') as f:
        for line in f:
            clinical_features.append(line.strip())

    return model, encoders, clinical_features

def main():
    """主函数 - 执行完整的数据处理和模型训练流程"""
    try:
        # 1. 下载并加载数据
        df = download_and_load_data()

        # 2. 数据预处理
        processed_df = preprocess_data(df)

        # 3. 创建交互矩阵
        interactions_df = create_interaction_matrix(processed_df)

        # 4. 编码特征
        encoded_interactions, patient_features_encoded, encoders = encode_features(
            processed_df, interactions_df
        )

        # 5. 准备模型输入
        train_data, val_data, test_data = prepare_model_inputs(encoded_interactions, patient_features_encoded)

        # 6. 分析数据
        analyze_data(processed_df, interactions_df)

        # 7. 构建并训练模型
        n_patients = len(encoders['patient_encoder'].classes_)
        n_drugs = len(encoders['drug_encoder'].classes_)
        n_factors = 32  # 嵌入维度
        n_patient_features = len(CLINICAL_FEATURES)

        model = build_ncf_model(n_patients, n_drugs, n_factors, n_patient_features)
        model, history = train_model(model, train_data, val_data)

        # 8. 评估模型
        evaluation_results = evaluate_model(model, test_data)

        # 9. 可视化训练历史
        visualize_training_history(history)

        # 10. 可视化推荐结果
        visualize_recommendations(model, encoders, patient_features_encoded, processed_df)

        # 11. 保存模型和工件
        save_model_and_artifacts(model, encoders)

        # 12. 保存处理后的数据
        processed_df.to_csv(f'{OUTPUT_DIR}/data/processed_diabetes_data.csv', index=False)
        interactions_df.to_csv(f'{OUTPUT_DIR}/data/diabetes_drug_interactions.csv', index=False)

        logger.info("数据处理和模型训练流程完成")

        # 13. 示例：为新患者生成推荐
        new_patient = {
            'A1Cresult': 2,  # >7
            'max_glu_serum': 2,  # >200
            'diag_1_category': DIAGNOSIS_CATEGORY_MAP['diabetes'],
            'diag_2_category': DIAGNOSIS_CATEGORY_MAP['cardiovascular'],
            'diag_3_category': DIAGNOSIS_CATEGORY_MAP['other'],
            'number_diagnoses': 3,
            'age_value': 65,
            'change': 1,
            'diabetesMed': 1,
            'num_medications': 5,
            'time_in_hospital': 7
        }

        recommendations = generate_recommendations_for_new_patient(new_patient, model, encoders)
        print("\n新患者的药物推荐:")
        print(recommendations)

        return {
            'processed_data': processed_df,
            'interactions': interactions_df,
            'model': model,
            'encoders': encoders,
            'evaluation_results': evaluation_results,
            'patient_features': patient_features_encoded
        }

    except Exception as e:
        logger.error(f"数据处理和模型训练过程中出错: {e}")
        import traceback
        traceback.print_exc()
        raise


if __name__ == "__main__":
    # 执行主流程
    result = main()


数据集基本信息:
总患者数: 56103
总记录数: 56103

药物使用情况:
                   drug  usage_count
8               insulin        37838
0             metformin        15371
4             glipizide         9331
5             glyburide         8049
6          pioglitazone         5424
7         rosiglitazone         4798
3           glimepiride         3826
1           repaglinide          965
9   glyburide-metformin          518
2           nateglinide          511
10             acarbose          209

diag_1_category 分布:
diabetes: 5153 (9.18%)
cardiovascular: 17228 (30.71%)
respiratory: 5627 (10.03%)
digestive: 4385 (7.82%)
genitourinary: 2682 (4.78%)
musculoskeletal: 3292 (5.87%)
injury: 3718 (6.63%)
anemia: 518 (0.92%)
mental: 1242 (2.21%)
other: 12258 (21.85%)

diag_2_category 分布:
diabetes: 7867 (14.02%)
cardiovascular: 17197 (30.65%)
respiratory: 5421 (9.66%)
digestive: 1927 (3.43%)
genitourinary: 4099 (7.31%)
musculoskeletal: 1032 (1.84%)
injury: 1926 (3.43%)
anemia: 1623 (2.89%)
mental: 1457 (2.60%)

3279/3279 ━━━━━━━━━━━━━━━━━━━━ 73s 21ms/step - accuracy: 0.8651 - loss: 0.3940 - val_accuracy: 0.8956 - val_loss: 0.2746 - learning_rate: 0.0010
Epoch 2/30
3279/3279 ━━━━━━━━━━━━━━━━━━━━ 73s 22ms/step - accuracy: 0.8969 - loss: 0.2630 - val_accuracy: 0.8989 - val_loss: 0.2815 - learning_rate: 0.0010
Epoch 3/30
3279/3279 ━━━━━━━━━━━━━━━━━━━━ 79s 21ms/step - accuracy: 0.9200 - loss: 0.1912 - val_accuracy: 0.8913 - val_loss: 0.3404 - learning_rate: 0.0010
Epoch 4/30
3279/3279 ━━━━━━━━━━━━━━━━━━━━ 87s 23ms/step - accuracy: 0.9345 - loss: 0.1497 - val_accuracy: 0.8706 - val_loss: 0.4233 - learning_rate: 0.0010
Epoch 5/30
3279/3279 ━━━━━━━━━━━━━━━━━━━━ 69s 21ms/step - accuracy: 0.9422 - loss: 0.1278 - val_accuracy: 0.8649 - val_loss: 0.5130 - learning_rate: 5.0000e-04
Epoch 6/30
3279/3279 ━━━━━━━━━━━━━━━━━━━━ 76s 23ms/step - accuracy: 0.9461 - loss: 0.1165 - val_accuracy: 0.8445 - val_loss: 0.5881 - learning_rate: 5.0000e-04
3858/3858 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.8954 - los

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 219ms/step

新患者的药物推荐:
           drug  probability
5       insulin     0.947799
6     metformin     0.163079
2     glipizide     0.110780
3     glyburide     0.075609
8  pioglitazone     0.064538
